In [16]:
!pip install -q groq pandas

import json
import pandas as pd
from getpass import getpass
from groq import Groq

print("Libraries imported successfully!")

Libraries imported successfully!


In [17]:
api_key = getpass("Enter your Groq API Key: ")

client = Groq(api_key=api_key)

MODEL = "openai/gpt-oss-120b"

print("Groq API configured successfully!")

Enter your Groq API Key: ··········
Groq API configured successfully!


In [18]:
TEACHER_PROMPT = """
You are an expert Python programming teacher.

Your role is to teach Python to complete beginners.

TEACHING RULES:

1. Explain concepts using simple English.
2. Avoid complicated technical terminology.
3. Explain concepts step by step.
4. Provide real-world examples.
5. Include simple Python code examples.
6. Explain code line by line.
7. Show expected output.
8. Guide students instead of simply giving answers.
9. Encourage students to practice.
10. Ask one practice question at the end.

TEACHING FORMAT:

1. Introduction
2. Real-world Example
3. Python Code
4. Code Explanation
5. Expected Output
6. Practice Question

TOPICS:

- Variables
- Data Types
- Operators
- Conditional Statements
- Loops
- Functions
- Lists
- Tuples
- Dictionaries

Always be friendly, clear and patient.
Adapt your explanation to the student's understanding.
"""

In [19]:
JUDGE_PROMPT = """
You are an independent LLM Teaching Evaluator.

Evaluate the response generated by the Python Teaching Agent.

EVALUATION CRITERIA:

1. conceptual_accuracy: 0 to 5
2. clarity: 0 to 5
3. beginner_friendliness: 0 to 5
4. example_quality: 0 to 5
5. teaching_approach: 0 to 5

SCORING:

0 = Completely missing or incorrect
1 = Very poor
2 = Needs improvement
3 = Satisfactory
4 = Good
5 = Excellent

EVALUATION RULES:

- Check technical accuracy.
- Check explanation clarity.
- Check beginner friendliness.
- Check code example quality.
- Check teaching approach.
- Evaluate honestly.
- Never give full marks automatically.

Return ONLY valid JSON.

Required structure:

{
    "conceptual_accuracy": 0,
    "clarity": 0,
    "beginner_friendliness": 0,
    "example_quality": 0,
    "teaching_approach": 0,
    "total_score": 0,
    "feedback": "",
    "suggestions": ""
}

The total_score must equal the sum of
the five individual scores.
"""

In [20]:
def python_teacher(question):

    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": TEACHER_PROMPT
            },
            {
                "role": "user",
                "content": question
            }
        ],

        temperature=0.4,
        max_tokens=1500
    )

    return response.choices[0].message.content

In [21]:
def teaching_judge(question, teacher_response):

    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": JUDGE_PROMPT
            },
            {
                "role": "user",
                "content": f"""
                Student Question:
                {question}

                Teacher Response:
                {teacher_response}

                Evaluate this response.
                """
            }
        ],

        response_format={"type": "json_object"},

        temperature=0.1,
        max_tokens=700
    )

    result = json.loads(
        response.choices[0].message.content
    )

    criteria = [
        "conceptual_accuracy",
        "clarity",
        "beginner_friendliness",
        "example_quality",
        "teaching_approach"
    ]

    # Validate scores
    for key in criteria:

        score = result[key]

        if isinstance(score, bool) or not isinstance(score, int) or not 0 <= score <= 5:
            raise ValueError(f"Invalid score: {key}")

    # Calculate score using Python
    result["total_score"] = sum(
        result[key] for key in criteria
    )

    return result

In [22]:
def run_teaching_agent(question):

    print("\n" + "=" * 60)
    print("STUDENT QUESTION")
    print("=" * 60)

    print(question)

    # First LLM: Teacher
    print("\nGenerating teaching response...")

    teacher_response = python_teacher(question)

    print("\n" + "=" * 60)
    print("PYTHON TEACHING AGENT")
    print("=" * 60)

    print(teacher_response)

    # Second LLM: Judge
    print("\nEvaluating teaching quality...")

    evaluation = teaching_judge(
        question,
        teacher_response
    )

    print("\n" + "=" * 60)
    print("LLM JUDGE EVALUATION")
    print("=" * 60)

    print(json.dumps(evaluation, indent=4))

    print("\nFINAL SCORE:",
          evaluation["total_score"], "/ 25")

    return {
        "question": question,
        "teacher_response": teacher_response,
        "evaluation": evaluation
    }

In [23]:
questions = [

    "What is a Python variable?",

    "Explain the difference between list and tuple.",

    "What is a for loop in Python? Explain with an example."

]

results = []

for question in questions:

    result = run_teaching_agent(question)

    results.append(result)


STUDENT QUESTION
What is a Python variable?

Generating teaching response...

PYTHON TEACHING AGENT
**1. Introduction**  
A *variable* in Python is like a labeled box that holds a value for you.  
You give the box a name, put something inside (a number, text, etc.), and later you can use that name to get the value back or change it.

**2. Real‑world Example**  
Imagine you are keeping track of the number of apples you have.  
You could write on a piece of paper:

```
apples = 5
```

Here **apples** is the label (the variable name) and **5** is the value stored in that “box”.  
If you later buy 2 more apples, you can update the box:

```
apples = apples + 2   # now the box holds 7
```

**3. Python Code**

```python
# create a variable called age and give it a value
age = 20

# show the value on the screen
print(age)

# change the value stored in the same variable
age = age + 5
print(age)
```

**4. Code Explanation (line by line)**  

| Line | What it does | Why it matters |
|------|---

In [25]:
evaluation_data = []

for result in results:

    evaluation = result["evaluation"]

    evaluation_data.append({

        "Question": result["question"],
        "Accuracy": evaluation["conceptual_accuracy"],
        "Clarity": evaluation["clarity"],
        "Beginner Friendly": evaluation["beginner_friendliness"],
        "Example Quality": evaluation["example_quality"],
        "Teaching Approach": evaluation["teaching_approach"],
        "Total Score": evaluation["total_score"]

    })

df = pd.DataFrame(evaluation_data)

display(df)

,Question,Accuracy,Clarity,Beginner Friendly,Example Quality,Teaching Approach,Total Score
0,What is a Python variable?,5,5,5,5,5,25
1,Explain the difference between list and tuple.,5,5,5,5,5,25
2,What is a for loop in Python? Explain with an ...,5,5,5,4,5,24


In [26]:
average_score = df["Total Score"].mean()

print(
    "Average Teaching Score:",
    round(average_score, 2),
    "/ 25"
)

Average Teaching Score: 24.67 / 25
